# Lab 14-02 · Instruction Set Architecture: RISC-V, Arm and x86

Companion to [chapter 14-02](../../chapters/14-02-isa-risc-v-arm-x86.md). Four parts: an RV32I encoder and decoder checked against the LLVM assembler; building constants with LUI and ADDI; code size in four ISAs; and a tiny RV32I simulator that runs the assembled machine code. You need `llvm-mc` (see the [lab handout](README.md)). Run the cells in order.

In [1]:
import glob, random, re, shutil, subprocess, tempfile
from pathlib import Path

def tool(name):
    found = [g for pat in ('/usr/lib/llvm-*/bin/', '/opt/homebrew/opt/llvm/bin/', '/usr/local/opt/llvm/bin/')
             for g in sorted(glob.glob(pat + name))]
    p = shutil.which(name) or next(iter(found), None)
    assert p, f'{name} not found: install LLVM (see README)'
    return p

MC, OBJCOPY = tool('llvm-mc'), tool('llvm-objcopy')
print(subprocess.run([MC, '--version'], capture_output=True, text=True).stdout.splitlines()[1].strip())

Optimized build.


## Part 1 · An encoder and a decoder from the format table

Each function below is one row of the format table in §3.3: it shifts each field into place. The B and J functions scatter the immediate's bits exactly as the table says.

In [2]:
REG = {f'x{i}': i for i in range(32)}
REG.update(zero=0, ra=1, sp=2, t0=5, t1=6, a0=10, a1=11)

def R(f7, rs2, rs1, f3, rd, op): return f7 << 25 | rs2 << 20 | rs1 << 15 | f3 << 12 | rd << 7 | op
def I(imm, rs1, f3, rd, op):     return (imm & 0xFFF) << 20 | rs1 << 15 | f3 << 12 | rd << 7 | op
def S(imm, rs2, rs1, f3, op):
    i = imm & 0xFFF
    return (i >> 5) << 25 | rs2 << 20 | rs1 << 15 | f3 << 12 | (i & 0x1F) << 7 | op
def B(imm, rs2, rs1, f3, op):
    i = imm & 0x1FFF
    return (i >> 12 & 1) << 31 | (i >> 5 & 0x3F) << 25 | rs2 << 20 | rs1 << 15 | f3 << 12 | (i >> 1 & 0xF) << 8 | (i >> 11 & 1) << 7 | op
def U(imm20, rd, op): return (imm20 & 0xFFFFF) << 12 | rd << 7 | op
def J(imm, rd, op):
    i = imm & 0x1FFFFF
    return (i >> 20 & 1) << 31 | (i >> 1 & 0x3FF) << 21 | (i >> 11 & 1) << 20 | (i >> 12 & 0xFF) << 12 | rd << 7 | op

def sx(x, bits): return x - (1 << bits) if x >> (bits - 1) & 1 else x

def decode(w):
    op, rd, f3, rs1, rs2, f7 = w & 0x7F, w >> 7 & 31, w >> 12 & 7, w >> 15 & 31, w >> 20 & 31, w >> 25
    if op in (0x13, 0x03, 0x67): imm = sx(w >> 20, 12)
    elif op == 0x23: imm = sx(f7 << 5 | rd, 12)
    elif op == 0x63: imm = sx((w >> 31 & 1) << 12 | (w >> 7 & 1) << 11 | (w >> 25 & 0x3F) << 5 | (w >> 8 & 0xF) << 1, 13)
    elif op in (0x37, 0x17): imm = w >> 12
    elif op == 0x6F: imm = sx((w >> 31 & 1) << 20 | (w >> 12 & 0xFF) << 12 | (w >> 20 & 1) << 11 | (w >> 21 & 0x3FF) << 1, 21)
    else: imm = None
    return dict(op=op, rd=rd, f3=f3, rs1=rs1, rs2=rs2, f7=f7, imm=imm)

def llvm_word(line):
    out = subprocess.run([MC, '-triple=riscv32', '-mattr=-c', '-show-encoding'], input=line,
                         capture_output=True, text=True, check=True).stdout
    toks = re.search(r'encoding: \[([^\]]*)\]', out).group(1).split(',')
    return int.from_bytes(bytes(int(t, 16) for t in toks), 'little')

Now encode a set of instructions both ways and compare. **Predict first:** what is the encoding of `sw x5, 12(x2)`? Work it out by hand from the S row of the table before running the cell.

In [3]:
tests = [
    ('add x5, x6, x7',     R(0x00, 7, 6, 0, 5, 0x33)),
    ('sub x5, x6, x7',     R(0x20, 7, 6, 0, 5, 0x33)),
    ('addi x5, x5, -273',  I(-273, 5, 0, 5, 0x13)),
    ('srai x5, x5, 3',     I(0x400 | 3, 5, 5, 5, 0x13)),
    ('lw x6, -4(x10)',     I(-4, 10, 2, 6, 0x03)),
    ('sw x5, 12(x2)',      S(12, 5, 2, 2, 0x23)),
    ('beq x5, x6, 16',     B(16, 6, 5, 0, 0x63)),
    ('bne x10, x0, -8',    B(-8, 0, 10, 1, 0x63)),
    ('bge x10, x11, -32',  B(-32, 11, 10, 5, 0x63)),
    ('lui x5, 0xDEADC',    U(0xDEADC, 5, 0x37)),
    ('auipc x6, 0x301',    U(0x301, 6, 0x17)),
    ('jal x1, 2048',       J(2048, 1, 0x6F)),
    ('jalr x1, -256(x6)',  I(-256, 6, 0, 1, 0x67)),
]
for text, mine in tests:
    ref = llvm_word(text)
    d = decode(mine)
    print(f'{text:20s} mine 0x{mine:08X}  llvm 0x{ref:08X}  {"ok" if mine == ref else "MISMATCH"}  decoded imm = {d["imm"]}')
assert all(mine == llvm_word(t) for t, mine in tests)

add x5, x6, x7       mine 0x007302B3  llvm 0x007302B3  ok  decoded imm = None
sub x5, x6, x7       mine 0x407302B3  llvm 0x407302B3  ok  decoded imm = None
addi x5, x5, -273    mine 0xEEF28293  llvm 0xEEF28293  ok  decoded imm = -273
srai x5, x5, 3       mine 0x4032D293  llvm 0x4032D293  ok  decoded imm = 1027
lw x6, -4(x10)       mine 0xFFC52303  llvm 0xFFC52303  ok  decoded imm = -4
sw x5, 12(x2)        mine 0x00512623  llvm 0x00512623  ok  decoded imm = 12
beq x5, x6, 16       mine 0x00628863  llvm 0x00628863  ok  decoded imm = 16
bne x10, x0, -8      mine 0xFE051CE3  llvm 0xFE051CE3  ok  decoded imm = -8
bge x10, x11, -32    mine 0xFEB550E3  llvm 0xFEB550E3  ok  decoded imm = -32
lui x5, 0xDEADC      mine 0xDEADC2B7  llvm 0xDEADC2B7  ok  decoded imm = 912092
auipc x6, 0x301      mine 0x00301317  llvm 0x00301317  ok  decoded imm = 769
jal x1, 2048         mine 0x001000EF  llvm 0x001000EF  ok  decoded imm = 2048
jalr x1, -256(x6)    mine 0xF00300E7  llvm 0xF00300E7  ok  decoded imm =

## Part 2 · Building constants

The split rule is hi = ⌊(C + 0x800)/4096⌋ mod 2²⁰ and lo = C − 4096·hi, with lo read as a signed 12-bit value. Part 4's simulator executes the pair for us. Here we check the rule on random constants and count how often the upper part must be rounded up.

In [4]:
def split(C):
    hi = ((C + 0x800) >> 12) & 0xFFFFF
    lo = sx((C - (hi << 12)) & 0xFFF, 12)
    return hi, lo

def rebuild(hi, lo):            # what LUI then ADDI leave in the register
    return ((hi << 12) + lo) & 0xFFFFFFFF

for C in (0xDEADBEEF, 0x12345678, 0x00000800, 0xFFFFF800, 0x7FFFF800):
    hi, lo = split(C)
    print(f'0x{C:08X}: lui 0x{hi:05X}, addi {lo:6d}  -> 0x{rebuild(hi, lo):08X}')

random.seed(2)
assert all(rebuild(*split(C)) == C for C in (random.getrandbits(32) for _ in range(100000)))
need = sum(split(low)[0] == 1 for low in range(4096))
print(f'{need} of 4096 low halves ({100 * need / 4096:.0f}%) need the upper part rounded up')

0xDEADBEEF: lui 0xDEADC, addi   -273  -> 0xDEADBEEF
0x12345678: lui 0x12345, addi   1656  -> 0x12345678
0x00000800: lui 0x00001, addi  -2048  -> 0x00000800
0xFFFFF800: lui 0x00000, addi  -2048  -> 0xFFFFF800
0x7FFFF800: lui 0x80000, addi  -2048  -> 0x7FFFF800
2048 of 4096 low halves (50%) need the upper part rounded up


## Part 3 · Code size in four ISAs

Assemble the array-sum function for RV32I with and without compression, A64 and x86-64, and list every instruction's bytes.

**Predict first:** which version is smallest, and which is largest?

LLVM prints the standard aliases: `addi t0, zero, 0` appears as `li t0, 0`, `beq a1, zero, done` as `beqz a1, done`, `jal zero, loop` as `j loop`, `addi a0, t0, 0` as `mv a0, t0` and `jalr zero, 0(ra)` as `ret`. The machine code is the same.

In [5]:
def assemble(path, triple, mattr=None):
    with tempfile.TemporaryDirectory() as td:
        obj, raw = Path(td) / 'a.o', Path(td) / 'a.bin'
        cmd = [MC, f'-triple={triple}', '-filetype=obj', str(path), '-o', str(obj)]
        if mattr: cmd.insert(2, f'-mattr={mattr}')
        subprocess.run(cmd, check=True)
        subprocess.run([OBJCOPY, '-O', 'binary', '--only-section=.text', str(obj), str(raw)], check=True)
        return raw.read_bytes()

def listing(path, triple, mattr=None):
    cmd = [MC, f'-triple={triple}', '-show-encoding'] + ([f'-mattr={mattr}'] if mattr else [])
    out = subprocess.run(cmd, input=Path(path).read_text(), capture_output=True, text=True, check=True).stdout
    for line in out.splitlines():
        m = re.match(r'\s+(\S.*?)\s+(?:#|//) encoding: \[([^\]]*)\]', line)
        if m:
            print(f'    {m.group(1).replace(chr(9), " "):28s} {len(m.group(2).split(","))} bytes')

for name, f, triple, mattr in (('RV32I', 'sum_rv32.s', 'riscv32', '-c,-relax'), ('RV32IC', 'sum_rv32.s', 'riscv32', '+c,-relax'),
                               ('A64', 'sum_a64.s', 'aarch64', None), ('x86-64', 'sum_x86.s', 'x86_64', None)):
    print(f'{name}: {len(assemble(f, triple, mattr))} bytes')
    listing(f, triple, mattr.split(',')[0] if mattr else None)

RV32I: 36 bytes
    li t0, 0                     4 bytes
    beqz a1, done                4 bytes
    lw t1, 0(a0)                 4 bytes
    add t0, t0, t1               4 bytes
    addi a0, a0, 4               4 bytes
    addi a1, a1, -1              4 bytes
    j loop                       4 bytes
    mv a0, t0                    4 bytes
    ret                          4 bytes
RV32IC: 20 bytes
    li t0, 0                     2 bytes
    beqz a1, done                2 bytes
    lw t1, 0(a0)                 4 bytes
    add t0, t0, t1               2 bytes
    addi a0, a0, 4               2 bytes
    addi a1, a1, -1              2 bytes
    j loop                       2 bytes
    mv a0, t0                    2 bytes
    ret                          2 bytes
A64: 32 bytes
    mov w2, wzr                  4 bytes
    cbz x1, done                 4 bytes
    ldr w3, [x0], #4             4 bytes
    add w2, w2, w3               4 bytes
    sub x1, x1, #1               4 bytes
    b loop

## Part 4 · A tiny RV32I simulator

This simulator holds 32 registers, a PC and a byte-addressed memory. Each step it fetches a word, decodes it with Part 1's decoder and executes it. It covers the base instructions these programs use. We load the assembler's own machine code, so a mistake in the decoder would show up as a wrong answer.

In [6]:
HALT, M32 = 0xFFFFFFF0, 0xFFFFFFFF

def run(code, regs, mem, base=0x1000):
    x = [0] * 32
    for k, val in regs.items(): x[REG[k]] = val & M32
    x[1] = HALT
    mem = dict(mem); mem.update({base + i: b for i, b in enumerate(code)})
    word = lambda a: int.from_bytes(bytes(mem.get(a + i, 0) for i in range(4)), 'little')
    s = lambda u: sx(u, 32)
    pc, n = base, 0
    while pc != HALT:
        d = decode(word(pc)); op, imm, a, b = d['op'], d['imm'], x[d['rs1']], x[d['rs2']]
        nxt, res = pc + 4, None
        if op == 0x37: res = imm << 12                                       # LUI
        elif op == 0x17: res = pc + (imm << 12)                              # AUIPC
        elif op == 0x6F: res, nxt = pc + 4, pc + imm                         # JAL
        elif op == 0x67: res, nxt = pc + 4, (a + imm) & ~1                   # JALR
        elif op == 0x63:                                                     # branches
            take = {0: a == b, 1: a != b, 4: s(a) < s(b), 5: s(a) >= s(b), 6: a < b, 7: a >= b}[d['f3']]
            if take: nxt = pc + imm
        elif op == 0x03: res = word((a + imm) & M32)                         # LW
        elif op == 0x13: res = {0: a + imm, 4: a ^ imm, 6: a | imm, 7: a & imm}[d['f3']]  # ADDI, XORI, ORI, ANDI
        elif op == 0x33: res = (a - b) if d['f7'] == 0x20 else {0: a + b, 3: int(a < b), 4: a ^ b, 6: a | b, 7: a & b}[d['f3']]
        else: raise ValueError(f'unsupported instruction 0x{word(pc):08X}')
        if res is not None and d['rd']: x[d['rd']] = res & M32
        pc, n = nxt & M32, n + 1
    return x, n

def array_mem(values, addr=0x10000):
    return {addr + 4 * i + k: (v & M32) >> 8 * k & 0xFF for i, v in enumerate(values) for k in range(4)}

data = list(range(1, 101))
for f in ('sum_rv32.s', 'sum_rv32_rot.s'):
    x, n = run(assemble(f, 'riscv32', '-c,-relax'), {'a0': 0x10000, 'a1': len(data)}, array_mem(data))
    print(f'{f:16s} returns {sx(x[10], 32)} after {n} instructions')

sum_rv32.s       returns 5050 after 604 instructions


sum_rv32_rot.s   returns 5050 after 504 instructions


**Explore.** Count the instructions for several array lengths and fit a straight line: you should find $6n + 4$ and $5n + 4$ (Problem 14-02.9). Then hand-assemble `lui x5, 0xDEADC` and `addi x5, x5, -273` with Part 1's functions, append `jalr x0, 0(x1)`, run them, and check that x5 ends up as 0xDEADBEEF.

In [7]:
for n_el in (0, 1, 10, 50):
    d = list(range(n_el))
    counts = [run(assemble(f, 'riscv32', '-c,-relax'), {'a0': 0x10000, 'a1': n_el}, array_mem(d))[1] for f in ('sum_rv32.s', 'sum_rv32_rot.s')]
    print(f'n = {n_el:3d}: top-tested {counts[0]:4d}, rotated {counts[1]:4d}')
prog = [U(0xDEADC, 5, 0x37), I(-273, 5, 0, 5, 0x13), I(0, 1, 0, 0, 0x67)]
x, _ = run(b''.join(w.to_bytes(4, 'little') for w in prog), {}, {})
print(f'x5 = 0x{x[5]:08X}')

n =   0: top-tested    4, rotated    4
n =   1: top-tested   10, rotated    9


n =  10: top-tested   64, rotated   54


n =  50: top-tested  304, rotated  254
x5 = 0xDEADBEEF
